In [1]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

In [2]:
raw = pd.read_csv('credit_card_cleaned.csv')

In [3]:
## create new columns for pay amounts / bill amounts

In [4]:
raw.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_REPAY_STATUS', 'AUG_REPAY_STATUS', 'JUL_REPAY_STATUS',
       'JUN_REPAY_STATUS', 'MAY_REPAY_STATUS', 'APR_REPAY_STATUS',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT',
       'default.payment.next.month', 'AGE_GROUP', 'EDUCATION_LEVEL',
       'MARRIAGE_DESC', 'LIMIT_BIN'],
      dtype='str')

In [5]:
bill_amount = ['SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT']
for col in bill_amount:
    print((raw[col]<0).sum())
    print('new line')
    print((raw[col]==0).sum())

# so like ~10% are 0 or below

590
new line
2008
669
new line
2506
655
new line
2870
675
new line
3195
655
new line
3506
688
new line
4020


In [6]:
payment_amount = ['SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT']
for col in payment_amount:
    print((raw[col]<0).sum())

# payments are never negative

0
0
0
0
0
0


### Pay amount / Bill amount

In [7]:
def add_payment_ratio(df):
    df = df.copy()
    pairs = [('SEP_PAY_AMT', 'AUG_BILL_AMT'),
             ('AUG_PAY_AMT', 'JUL_BILL_AMT'),
             ('JUL_PAY_AMT', 'JUN_BILL_AMT'),
             ('JUN_PAY_AMT', 'MAY_BILL_AMT'),
             ('MAY_PAY_AMT', 'APR_BILL_AMT')]
    # have to exclude sep :(
    
    for payment_col, prior_bill_col in pairs:
        name = 'PAY_RATIO_' + payment_col[:3]
        r = df[payment_col] / df[prior_bill_col]
        r = r.replace([np.inf, -np.inf], np.nan)
        r = r.mask(df[prior_bill_col] <= 0, np.nan)
        r = r.clip(lower=0.0, upper=2.0)
        df[name] = r
    return df

In [8]:
feat = add_payment_ratio(raw)


### Balance / Limit (utilization rate)

In [9]:
def add_utilization_rate(df):
    df = df.copy()
    balances = ['SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT']
    
    for bill_col in balances:
        name = 'UTIL_' + bill_col[:3]
        df[name] = df[bill_col] / df['LIMIT_BAL']
    return df

In [10]:
feat = add_utilization_rate(feat)

In [11]:
feat.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE',
       'SEP_REPAY_STATUS', 'AUG_REPAY_STATUS', 'JUL_REPAY_STATUS',
       'JUN_REPAY_STATUS', 'MAY_REPAY_STATUS', 'APR_REPAY_STATUS',
       'SEP_BILL_AMT', 'AUG_BILL_AMT', 'JUL_BILL_AMT', 'JUN_BILL_AMT',
       'MAY_BILL_AMT', 'APR_BILL_AMT', 'SEP_PAY_AMT', 'AUG_PAY_AMT',
       'JUL_PAY_AMT', 'JUN_PAY_AMT', 'MAY_PAY_AMT', 'APR_PAY_AMT',
       'default.payment.next.month', 'AGE_GROUP', 'EDUCATION_LEVEL',
       'MARRIAGE_DESC', 'LIMIT_BIN', 'PAY_RATIO_SEP', 'PAY_RATIO_AUG',
       'PAY_RATIO_JUL', 'PAY_RATIO_JUN', 'PAY_RATIO_MAY', 'UTIL_SEP',
       'UTIL_AUG', 'UTIL_JUL', 'UTIL_JUN', 'UTIL_MAY', 'UTIL_APR'],
      dtype='str')

In [12]:
print((feat['UTIL_SEP'] > 1).sum())
print(feat[feat['UTIL_SEP'] > 1]['default.payment.next.month'].mean())

2115
0.300709219858156


### Aggregates over utiilzation rate (good for deterining financial risk)

In [13]:
def add_utilization_agg(df):
    df = df.copy()
    util_cols = ['UTIL_SEP', 'UTIL_AUG', 'UTIL_JUL',
       'UTIL_JUN', 'UTIL_MAY', 'UTIL_APR']
    df['UTIL_AVG'] = df[util_cols].mean(axis=1)
    df['UTIL_MAX'] = df[util_cols].max(axis=1)
    df['UTIL_MIN'] = df[util_cols].min(axis=1)
    df['UTIL_STD'] = df[util_cols].std(axis=1)
    df['MONTHS_NO_BALANCE'] = (df[util_cols]==0).sum(axis=1)
    r = df['SEP_BILL_AMT'] / df['APR_BILL_AMT']
    r = r.replace([np.inf, -np.inf], np.nan)
    r = r.mask(df['APR_BILL_AMT'] <= 0)
    
    df['BILL_GROWTH_RATIO'] = r
    df['UTIL_TREND'] = df['UTIL_SEP'] - df['UTIL_APR']
    df['PAY_TREND'] = df['PAY_RATIO_SEP'] - df['PAY_RATIO_MAY']
    return df

In [14]:
feat = add_utilization_agg(feat)

In [15]:
print(feat.head())

   ID  LIMIT_BAL  SEX  EDUCATION  MARRIAGE  AGE  SEP_REPAY_STATUS  \
0   1    20000.0    2          2         1   24                 2   
1   2   120000.0    2          2         2   26                -1   
2   3    90000.0    2          2         2   34                 0   
3   4    50000.0    2          2         1   37                 0   
4   5    50000.0    1          2         1   57                -1   

   AUG_REPAY_STATUS  JUL_REPAY_STATUS  JUN_REPAY_STATUS  ...  UTIL_MAY  \
0                 2                -1                -1  ...  0.000000   
1                 2                 0                 0  ...  0.028792   
2                 0                 0                 0  ...  0.166089   
3                 0                 0                 0  ...  0.579180   
4                 0                -1                 0  ...  0.382920   

   UTIL_APR  UTIL_AVG  UTIL_MAX  UTIL_MIN  UTIL_STD  MONTHS_NO_BALANCE  \
0  0.000000  0.064200  0.195650  0.000000  0.088082               

In [16]:
feat.to_csv('credit_card_featured.csv', index=False)